Data prepration


In [ ]:
"""
This script filters the COCO dataset to include only the selected 11 food classes.
It copies both the images and their corresponding YOLO-format labels to a new folder
for focused training.
"""


import os
import shutil
# Path to the original COCO training dataset
dataset_path = r"C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\archive\train"

# Path where the filtered dataset will be saved
output_path = "dataset_5_classes_train"

# List of class IDs to keep (corresponding to the food classes you want)
keep_classes = [1, 3, 15, 19, 17, 28, 37, 45, 49, 53, 78]

# Create output directories for images and labels if they don't already exist
os.makedirs(os.path.join(output_path, "images"), exist_ok=True)
os.makedirs(os.path.join(output_path, "labels"), exist_ok=True)

# Loop through all label files in the original dataset
for label_file in os.listdir(os.path.join(dataset_path, "labels")):
    # Open the label file
    with open(os.path.join(dataset_path, "labels", label_file)) as f:
        lines = f.readlines()  # Read all annotation lines

    # Filter annotations: keep only objects with class IDs in 'keep_classes'
    filtered = [line for line in lines if int(line.split()[0]) in keep_classes]

    # If there is at least one object of interest in the image
    if filtered:
        # Save the filtered annotations to the new labels folder
        with open(os.path.join(output_path, "labels", label_file), "w") as f:
            f.writelines(filtered)

        # Copy the corresponding image to the new images folder
        img_name = label_file.replace(".txt", ".jpg")  # Change label filename to image filename
        shutil.copy(
            os.path.join(dataset_path, "images", img_name),
            os.path.join(output_path, "images", img_name)
        )

# Result:
# - 'dataset_5_classes_train/images/' contains only images with the selected classes
# - 'dataset_5_classes_train/labels/' contains only the corresponding label files
# - This filtered dataset can now be used for training YOLO on only the desired classes


In [ ]:
"""
This script remaps the original COCO class IDs in YOLO label files to a new sequential
set of class IDs for a smaller subset of classes. 

- The original COCO dataset has 80 classes, but the project focuses only on 11 food classes.
- This mapping assigns new IDs (0–10) to these selected classes, which YOLO expects.
- For each label file:
    1. Read all annotation lines.
    2. Keep only the objects belonging to the selected classes.
    3. Replace the original class ID with the new sequential ID.
    4. Save the updated label file to a new folder.
- This ensures that the YOLO training process uses consecutive class IDs and ignores unwanted classes.
"""

import os

# Your paths
labels_dir = "dataset/train/labels"  # change to val if needed
output_dir = "new_labels"
os.makedirs(output_dir, exist_ok=True)

# COCO -> new mapping
old_to_new = {
  1: 0,
  3: 1,
  15: 2,
  19: 3,
  17: 4,
  28: 5,
  37: 6,
  45: 7,
  49: 8,
  53: 9,
  78: 10
}

for file in os.listdir(labels_dir):
    if not file.endswith(".txt"):
        continue
    
    input_path = os.path.join(labels_dir, file)
    output_path = os.path.join(output_dir, file)
    
    new_lines = []
    with open(input_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            old_cls = int(parts[0])
            
            if old_cls in old_to_new:
                parts[0] = str(old_to_new[old_cls])
                new_lines.append(" ".join(parts))
    
    # Write only if object exists
    if new_lines:
        with open(output_path, "w") as f:
            f.write("\n".join(new_lines))



installing  ultralytics

In [7]:
!pip install ultralytics --upgrade

  Using cached ultralytics-8.3.179-py3-none-any.whl.metadata (37 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     ------ --------------------------------- 10.2/60.8 kB ? eta -:--:--
     ------------------------------- ------ 51.2/60.8 kB 660.6 kB/s eta 0:00:01
     -------------------------------------- 60.8/60.8 kB 648.3 kB/s eta 0:00:00
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/111.1 kB ? eta -:--:--
     ---------------------------- ---------- 81.9/111.1 kB 4.5 MB/s eta 0:00:01
     -------------------------------------- 111.1/111.1 kB 1.6 MB/s eta 0:00:00
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Usi


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Training

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8n model (Nano version, fast and lightweight)
# You can also use 'yolov8s.pt' for slightly larger model with better accuracy
model = YOLO("yolov8n.pt")


In [2]:
import torch
print(torch.cuda.is_available())  # True means GPU is ready
print(torch.cuda.get_device_name(0))  # Name of GPU


True
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [ ]:

# Start training
model.train(
    # Path to the dataset YAML file
    # The YAML file should contain paths to train/val images and class names
    data="C:\\Users\\sshak\\OneDrive\\Desktop\\codes\\cocc2017\\dataset\\data.yaml",

    # Number of epochs to train for
    epochs=70,  

    # Image size for training and validation (640x640 recommended)
    imgsz=640,  

    # Batch size per GPU (adjust based on your GPU memory)
    batch=16,  

    # Directory to save results, weights, and logs
    project="yolo11_classes",  

    # Name of this training run (folder inside project directory)
    name="exp1",  

    # Overwrite existing run if it has the same name
    exist_ok=True,  

    # Save model weights every 'n' epochs
    # Here, saving every epoch
    save_period=1,  

    # Device to train on: 0 for first GPU, 'cpu' for CPU training
    device=0  
)

# Notes:
# - 'yolov8n.pt' is a lightweight model suitable for smaller datasets or limited GPU.
# - You can adjust 'epochs' for longer training to improve mAP.
# - Batch size must fit in your GPU memory; reduce if CUDA out-of-memory error occurs.
# - The YAML file must follow YOLO format and include:
#       train: path/to/train/images
#       val: path/to/val/images
#       nc: number_of_classes
#       names: [list of class names]


Ultralytics 8.3.179  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

train: Scanning C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\train\labels.cache... 17577 images, 0 backgrounds, 0 corrupt: 100%|██████████| 17577/17577 [00:00<?, ?it/s]


validation

In [1]:
from ultralytics import YOLO

# Load your trained YOLOv8 model
# 'best.pt' is the checkpoint with the highest validation mAP
model = YOLO(r"best.pt")

# Evaluate the model on the validation dataset
# The .val() method calculates mAP, precision, recall, and other metrics
results = model.val(
    data=r"C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\data.yaml",  # path to dataset YAML
    batch=16,      # batch size for validation (adjust based on GPU memory)
    imgsz=640      # image size (should match training image size)
)

# Print evaluation metrics
# 'results' includes overall mAP, class-wise mAP, precision, recall, and confusion matrix
print(results)


Ultralytics 8.3.179  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
Model summary (fused): 72 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 109.442.3 MB/s, size: 52.9 KB)


val: Scanning C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\val\labels.cache... 774 images, 0 backgrounds, 0 corrupt: 100%|██████████| 774/774 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 49/49 [00:07<00:00,  6.20it/s]


                   all        774       3133      0.539      0.465      0.462       0.32
                 apple         76        236      0.444      0.305      0.287      0.206
                banana        103        370      0.462      0.378      0.371      0.233
              broccoli         71        312      0.473      0.383      0.388      0.213
                carrot         81        365      0.425      0.384      0.329      0.208
                  cake        124        310      0.564      0.523      0.535      0.348
                 donut         62        328      0.582      0.527      0.545      0.415
               hot dog         51        125      0.611       0.49      0.482      0.359
                orange         85        285      0.472      0.432      0.399      0.309
                 pizza        153        284      0.692      0.704      0.716      0.542
              sandwich         98        177      0.552      0.531      0.527      0.378
            wine glas

Inference


In [ ]:
from ultralytics import YOLO
import cv2
import os

# --- 1. Load your trained model ---
model = YOLO(r"best.pt")  # or last.pt

# --- 2. Set paths ---
test_images_folder = r"C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\val\images"  # folder containing test images
output_folder = r"C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\inference_results"
os.makedirs(output_folder, exist_ok=True)

# --- 3. Run inference on each image ---
for img_name in os.listdir(test_images_folder):
    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path = os.path.join(test_images_folder, img_name)
        
        # Run detection
        results = model.predict(source=img_path, conf=0.25, save=False)
        
        # Convert results to OpenCV image
        img = cv2.imread(img_path)
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()        # bounding boxes
            scores = result.boxes.conf.cpu().numpy()       # confidence scores
            classes = result.boxes.cls.cpu().numpy()       # class indices
            names = model.names                            # class names
            
            # Draw bounding boxes
            for box, score, cls in zip(boxes, scores, classes):
                x1, y1, x2, y2 = map(int, box)
                label = f"{names[int(cls)]}: {score:.2f}"
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (0, 255, 0), 2)
        
        # Save result
        save_path = os.path.join(output_folder, img_name)
        cv2.imwrite(save_path, img)
        print(f"Inference saved for {img_name} -> {save_path}")

print("✅ All images processed!")



image 1/1 C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\val\images\000000001425_jpg.rf.2272d6db1b833adc2797647f3073c629.jpg: 512x640 1 sandwich, 116.6ms
Speed: 11.4ms preprocess, 116.6ms inference, 36.0ms postprocess per image at shape (1, 3, 512, 640)
Inference saved for 000000001425_jpg.rf.2272d6db1b833adc2797647f3073c629.jpg -> C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\inference_results\000000001425_jpg.rf.2272d6db1b833adc2797647f3073c629.jpg

image 1/1 C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\val\images\000000002149_jpg.rf.f2ac2c2da1912ce0195965bd3d3729aa.jpg: 448x640 5 apples, 1 orange, 97.6ms
Speed: 3.9ms preprocess, 97.6ms inference, 6.1ms postprocess per image at shape (1, 3, 448, 640)
Inference saved for 000000002149_jpg.rf.f2ac2c2da1912ce0195965bd3d3729aa.jpg -> C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\inference_results\000000002149_jpg.rf.f2ac2c2da1912ce0195965bd3d3729aa.jpg

image 1/1 C:\Users\sshak\OneDrive\Desktop\codes\cocc2017\dataset\